In [1]:
"""
STEP 3: Chart-first hierarchy + a toggle that changes WHAT the hero chart shows.
New concept this step: a callback can react to MULTIPLE inputs (season + mode + team),
and one callback's output (the mode) can control whether another element is even visible.
"""

import dash
from dash import dcc, html, Input, Output, State
import dash_bootstrap_components as dbc
import plotly.graph_objects as go
import pandas as pd

import os
app = dash.Dash(__name__,
                 external_stylesheets=[dbc.themes.DARKLY],
                 assets_folder=os.path.join(os.getcwd(), 'assets'),
                 suppress_callback_exceptions=True)

# ---- 1. DATA SETUP ----

df_all = pd.read_csv('epl_final.csv')
df_all['MatchDate'] = pd.to_datetime(df_all['MatchDate'])

TOP3_BY_SEASON = {
    '2020/21': ['Man City', 'Man United', 'Liverpool'],
    '2021/22': ['Man City', 'Liverpool', 'Chelsea'],
    '2022/23': ['Man City', 'Arsenal', 'Man United'],
    '2023/24': ['Man City', 'Arsenal', 'Liverpool'],
    '2024/25': ['Liverpool', 'Arsenal', 'Man City'],
}

TEAM_COLORS = {
    'Man City': '#6CABDD', 'Man United': '#E63946', 'Liverpool': '#00D9A3',
    'Arsenal': '#EF0107', 'Chelsea': '#4CC9F0'
}

BG_DARK = '#0F1115'
PANEL_DARK = '#171A21'
TEXT_LIGHT = '#E8E8E8'
TEXT_MUTED = '#8A8F98'
ACCENT = '#00D9A3'


def build_team_sequence(df, team):
    """Full per-match sequence: Open/High/Low/Close momentum + EMA5, same formula as the original notebooks."""
    rows = []
    for _, r in df.iterrows():
        if r['HomeTeam'] == team:
            res = 1 if r['FullTimeResult'] == 'H' else (0 if r['FullTimeResult'] == 'D' else -1)
        elif r['AwayTeam'] == team:
            res = 1 if r['FullTimeResult'] == 'A' else (0 if r['FullTimeResult'] == 'D' else -1)
        else:
            continue
        rows.append({'Date': r['MatchDate'], 'Result': res})
    seq = pd.DataFrame(rows).sort_values('Date').reset_index(drop=True)

    level, last_color = 0.0, None
    opens, closes = [], []
    for _, row in seq.iterrows():
        open_price = level
        if row['Result'] == 1:
            close = open_price + 1.0; last_color = 'green'
        elif row['Result'] == -1:
            close = open_price - 1.0; last_color = 'red'
        else:
            if last_color == 'green':
                close = max(open_price - open_price * 0.5, 0)
            elif last_color == 'red':
                close = min(open_price + (0 - open_price) * 0.5, 0)
            else:
                close = 0.0
        opens.append(open_price); closes.append(close)
        level = close
    seq['Open'] = opens
    seq['Close'] = closes
    seq['High'] = seq[['Open', 'Close']].max(axis=1)
    seq['Low'] = seq[['Open', 'Close']].min(axis=1)
    seq['EMA5'] = seq['Close'].ewm(span=5, adjust=False).mean()
    seq['MatchNum'] = range(1, len(seq) + 1)
    return seq


def kpi_card(label, value, sub=""):
    return dbc.Card(
        dbc.CardBody([
            html.Div(label.upper(), style={'fontSize': '11px', 'color': TEXT_MUTED, 'letterSpacing': '1px'}),
            html.Div(value, style={'fontSize': '26px', 'color': TEXT_LIGHT, 'fontWeight': '600'}),
            html.Div(sub, style={'fontSize': '12px', 'color': TEXT_MUTED})
        ]),
        style={'backgroundColor': PANEL_DARK, 'border': '1px solid #262b33'}
    )


# ---- 2. APP ----

app = dash.Dash(__name__, external_stylesheets=[dbc.themes.DARKLY],
                 suppress_callback_exceptions=True)
# suppress_callback_exceptions is required because 'team-dropdown' doesn't exist in the
# initial layout — it's created dynamically by callback 1, so Dash needs to be told not
# to validate that ID against the layout at startup.

app.layout = html.Div(style={'backgroundColor': BG_DARK, 'minHeight': '100vh', 'padding': '30px'}, children=[

    html.Div([
        html.H2("Does early momentum predict who finishes on top?",
                style={'color': TEXT_LIGHT, 'fontWeight': '700', 'marginBottom': '4px'}),
        html.P("Match-by-match win/draw/loss momentum, smoothed with a 5-match EMA.",
               style={'color': TEXT_MUTED, 'fontSize': '14px'})
    ], style={'marginBottom': '20px'}),

    # Controls row: season + mode toggle + (conditionally) team picker, all feeding the hero chart
    dbc.Row([
        dbc.Col([
            html.Label("SEASON", style={'color': TEXT_MUTED, 'fontSize': '11px', 'letterSpacing': '1px'}),
            dcc.Dropdown(
                id='season-dropdown',
                options=[{'label': s, 'value': s} for s in TOP3_BY_SEASON.keys()],
                value='2021/22', clearable=False
            ),
        ], width=3),

        dbc.Col([
            html.Label("CHART MODE", style={'color': TEXT_MUTED, 'fontSize': '11px', 'letterSpacing': '1px'}),
            dcc.RadioItems(
                id='chart-mode',
                options=[
                    {'label': ' Top 3 Comparison', 'value': 'compare'},
                    {'label': ' Single Team Deep-Dive', 'value': 'deepdive'},
                ],
                value='compare',
                labelStyle={'display': 'block', 'color': TEXT_LIGHT, 'marginTop': '6px'},
                inputStyle={'marginRight': '6px'}
            )
        ], width=4),

        # Only shown when mode == 'deepdive' — controlled by the callback below
        dbc.Col(id='team-picker-container', width=3),
    ], style={'marginBottom': '20px'}),

    # dcc.Store holds the selected team as plain data — this is the standard Dash way
    # to pass a value between callbacks, instead of parsing rendered HTML children.
    dcc.Store(id='selected-team-store'),

    # HERO CHART — dominant, full width, first thing the eye lands on
    dcc.Graph(id='hero-chart', config={'displayModeBar': False},
              style={'height': '480px', 'marginBottom': '25px'}),

    # KPI row — supporting context, visually smaller and secondary
    dbc.Row(id='kpi-row'),
])


# Callback 1: show/hide the team dropdown based on chosen mode.
# This demonstrates one control (chart-mode) changing whether ANOTHER control even exists.
@app.callback(
    Output('team-picker-container', 'children'),
    Input('chart-mode', 'value'),
    Input('season-dropdown', 'value')
)
def show_team_picker(mode, season):
    if mode != 'deepdive':
        return []  # empty = nothing rendered
    teams = TOP3_BY_SEASON[season]
    return [
        html.Label("TEAM", style={'color': TEXT_MUTED, 'fontSize': '11px', 'letterSpacing': '1px'}),
        dcc.Dropdown(id='team-dropdown',
                     options=[{'label': t, 'value': t} for t in teams],
                     value=teams[0], clearable=False)
    ]


# Callback 2: whenever the team-dropdown exists and changes, push its value into the Store.
# This fires only when 'team-dropdown' is actually on the page (deepdive mode).
@app.callback(
    Output('selected-team-store', 'data'),
    Input('team-dropdown', 'value'),
    prevent_initial_call=False
)
def store_selected_team(team_value):
    return team_value


# Callback 3: the hero chart + KPIs, reacting to season, mode, AND (when relevant) the stored team.
@app.callback(
    Output('hero-chart', 'figure'),
    Output('kpi-row', 'children'),
    Input('season-dropdown', 'value'),
    Input('chart-mode', 'value'),
    Input('selected-team-store', 'data'),
)
def update_hero(selected_season, mode, stored_team):
    season_df = df_all[df_all['Season'] == selected_season]
    fig = go.Figure()

    if mode == 'compare':
        teams = TOP3_BY_SEASON[selected_season]
        final_emas = {}
        for team in teams:
            seq = build_team_sequence(season_df, team)
            final_emas[team] = seq['EMA5'].iloc[-1]
            fig.add_trace(go.Scatter(
                x=seq['MatchNum'], y=seq['EMA5'], mode='lines', name=team,
                line=dict(color=TEAM_COLORS.get(team, '#888'), width=3)
            ))
        leader = max(final_emas, key=final_emas.get)
        gap = round(max(final_emas.values()) - sorted(final_emas.values())[-2], 2)
        kpis = [
            dbc.Col(kpi_card("Season leader", leader, f"EMA-5 = {round(final_emas[leader],2)}"), width=4),
            dbc.Col(kpi_card("Gap to 2nd place", f"{gap}", "momentum points"), width=4),
            dbc.Col(kpi_card("Teams tracked", str(len(teams)), ", ".join(teams)), width=4),
        ]
        title = f"Top 3 Momentum — EMA(5) | {selected_season}"

    else:  # deepdive
        teams_this_season = TOP3_BY_SEASON[selected_season]
        # fall back to the first team if the store is empty or holds a team not in this season
        team = stored_team if stored_team in teams_this_season else teams_this_season[0]

        seq = build_team_sequence(season_df, team)
        fig.add_trace(go.Candlestick(
            x=seq['MatchNum'], open=seq['Open'], high=seq['High'], low=seq['Low'], close=seq['Close'],
            increasing_line_color=ACCENT, decreasing_line_color='#E63946', name=team, showlegend=False
        ))
        fig.add_trace(go.Scatter(
            x=seq['MatchNum'], y=seq['EMA5'], mode='lines', name='EMA(5)',
            line=dict(color='#FFD166', width=2, dash='dot')
        ))
        peak_ema = seq['EMA5'].max()
        final_ema = seq['EMA5'].iloc[-1]
        wins = (seq['Close'].diff() == 1).sum()
        kpis = [
            dbc.Col(kpi_card("Team", team, f"{len(seq)} matches"), width=4),
            dbc.Col(kpi_card("Peak EMA(5)", round(peak_ema, 2), "highest momentum reached"), width=4),
            dbc.Col(kpi_card("Final EMA(5)", round(final_ema, 2), "momentum at season close"), width=4),
        ]
        title = f"{team} Momentum Deep-Dive | {selected_season}"

    fig.update_layout(
        template='plotly_dark', plot_bgcolor=PANEL_DARK, paper_bgcolor=PANEL_DARK,
        font=dict(color=TEXT_LIGHT), title=title,
        xaxis_title="Match Number", yaxis_title="Momentum",
        margin=dict(t=50)
    )
    return fig, kpis


if __name__ == '__main__':
    #app.run(debug=True)
    # In Jupyter: comment the line above, use:
     app.run(jupyter_mode="inline", debug=True)

Error on request:
Traceback (most recent call last):
  File "C:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\werkzeug\serving.py", line 370, in run_wsgi
    execute(self.server.app)
  File "C:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\werkzeug\serving.py", line 355, in execute
    data = self.rfile.read(10_000_000)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
MemoryError


In [10]:
import os
print(os.getcwd())

C:\Users\user\Downloads


In [11]:
import os
print(os.path.exists('assets/custom.css'))
print(os.path.getsize('assets/custom.css'), 'bytes')

True
1012 bytes


In [8]:
print("Assets folder Dash is using:", app.config.assets_folder)
print("Files in that folder:", os.listdir(app.config.assets_folder))

client = app.server.test_client()
response = client.get('/')
html = response.get_data(as_text=True)
print("Is custom.css linked in the page?", 'custom.css' in html)

Assets folder Dash is using: C:\Users\user\Downloads\assets
Files in that folder: ['custom.css']
Is custom.css linked in the page? True


In [9]:
import dash
print("Your Dash version:", dash.__version__)

Your Dash version: 3.0.4
